# 03 Content-Based Recommendation Engine & Cold-Start Validation

Այս նոթբուքում իրականացված է **Cosine Similarity** վրա հիմնված Content-Based Recommendation Engine-ը:
- Օգտագործվում է նախորդ փուլում ստացված **56-չափանի Feature Matrix-ը**:
- Հաշվարկը կատարվում է **On-Demand** (ըստ պահանջի)՝ RAM-ի գերբեռնումից խուսափելու համար:
- Ստուգվում է **Cold-Start** սցենարը (0 review ունեցող listing-ների համար):

In [1]:
import pandas as pd
from recsys_utils import fix_text

p = '../data/processed/listings_clean.pkl'
df = pd.read_pickle(p)
df['name'] = df['name'].map(fix_text)
df.to_pickle(p)
print(df['name'].head(10).tolist())

['Beautiful Flat in le Village Montmartre, Paris', '39 m² Paris (Sacre Cœur)', 'Lovely apartment with Terrace, 60m2', 'Cosy studio (close to Eiffel tower)', 'Close to Eiffel Tower - Beautiful flat : 2 rooms', 'NEW - Charming apartment Le Marais', '2P - Entre Bastille et Republique', '57sqm btw. Bastille & Père Lachaise', 'Charming appartment near the Parc Buttes Chaumont', 'Bel appartement plein de charme !']


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

item_features = pd.read_pickle('../data/processed/item_features.pkl')
listings_clean = pd.read_pickle('../data/processed/listings_clean.pkl')

print(f"Loaded Item Features Shape: {item_features.shape}")   
print(f"Loaded Listings Shape: {listings_clean.shape}")       

def get_content_recommendations(target_id, item_matrix, listings_df, top_n=10):
    id_col = 'listing_id' if 'listing_id' in listings_df.columns else 'id'
    matches = np.where(listings_df[id_col].values == target_id)[0]
    if len(matches) == 0:
        print(f"ID {target_id} չի գտնվել:")
        return None
    pos = matches[0]

    sim_scores = cosine_similarity(item_matrix.iloc[[pos]].values, item_matrix.values)[0]

    
    same_city = listings_df['city'].values == listings_df['city'].values[pos]
    sim_scores[~same_city] = -1
    sim_scores[pos] = -1          

    top_indices = np.argsort(sim_scores)[::-1][:top_n]
    rec_df = listings_df.iloc[top_indices].copy()
    rec_df['similarity_score'] = sim_scores[top_indices]

    cols = [id_col, 'city', 'name', 'room_type', 'price', 'bedrooms', 'neighbourhood', 'similarity_score']
    return rec_df[[c for c in cols if c in rec_df.columns]]

Loaded Item Features Shape: (279599, 56)
Loaded Listings Shape: (279599, 37)


In [3]:
id_col = 'listing_id' if 'listing_id' in listings_clean.columns else 'id'
sample_id = listings_clean[id_col].iloc[0]

print(f"\n=== RECOMMENDATIONS FOR SAMPLE LISTING ({id_col.upper()}: {sample_id}) ===")

cols_to_show = [id_col, 'name', 'room_type', 'price', 'bedrooms', 'neighbourhood']
existing_cols = [c for c in cols_to_show if c in listings_clean.columns]

target_info = listings_clean[listings_clean[id_col] == sample_id][existing_cols]
print("\n--- Target Listing Info ---")
display(target_info)

print("\n--- Top 5 Recommended Listings ---")
display(get_content_recommendations(sample_id, item_features, listings_clean, top_n=5))


=== RECOMMENDATIONS FOR SAMPLE LISTING (LISTING_ID: 281420) ===

--- Target Listing Info ---


,listing_id,name,room_type,price,bedrooms,neighbourhood
0,281420,"Beautiful Flat in le Village Montmartre, Paris",Entire place,53,1.0,Buttes-Montmartre



--- Top 5 Recommended Listings ---


,listing_id,city,name,room_type,price,bedrooms,neighbourhood,similarity_score
6705,7336461,Paris,"Studio près du Sacre Coeur, Paris",Entire place,55,1.0,Buttes-Montmartre,0.999778
278173,526920,Paris,Montmartre at the best price !,Entire place,50,1.0,Buttes-Montmartre,0.999470
275873,8917189,Paris,studio,Entire place,35,1.0,Buttes-Montmartre,0.977068
278312,578990,Paris,Very nice flat close to Montmartre,Entire place,35,1.0,Buttes-Montmartre,0.977068
277949,741426,Paris,CHARME NEAR MONTMARTRE,Entire place,80,1.0,Buttes-Montmartre,0.969907


In [4]:
if 'number_of_reviews' in listings_clean.columns:
    cold_listings = listings_clean[listings_clean['number_of_reviews'] == 0]
    
    if len(cold_listings) > 0:
        cold_id = cold_listings[id_col].iloc[0]
        print(f"\n=== COLD-START TEST FOR LISTING WITH 0 REVIEWS ({id_col.upper()}: {cold_id}) ===")
        
        cols_to_show = [id_col, 'name', 'room_type', 'price', 'bedrooms', 'neighbourhood', 'number_of_reviews']
        existing_cols = [c for c in cols_to_show if c in cold_listings.columns]
        
        target_cold_info = cold_listings[cold_listings[id_col] == cold_id][existing_cols]
        print("\n--- Cold Target Listing Info ---")
        display(target_cold_info)
        
        print("\n--- Content-Based Recommendations for Cold Listing ---")
        display(get_content_recommendations(cold_id, item_features, listings_clean, top_n=5))


=== COLD-START TEST FOR LISTING WITH 0 REVIEWS (LISTING_ID: 5469152) ===

--- Cold Target Listing Info ---


,listing_id,name,room_type,price,bedrooms,neighbourhood,number_of_reviews
1990,5469152,bel appartement paris Montparnasse,Entire place,59,1.0,Observatoire,0



--- Content-Based Recommendations for Cold Listing ---


,listing_id,city,name,room_type,price,bedrooms,neighbourhood,similarity_score
6825,7721117,Paris,Studio 18m2 proche de Bercy,Entire place,50,1.0,Reuilly,0.996071
279009,547907,Paris,Cosy apartment / Center of Paris,Entire place,55,1.0,Bourse,0.942582
6716,7727343,Paris,Spacious Studio in Paris Center,Entire place,59,1.0,Bourse,0.942308
7053,8226993,Paris,Le petit pernety Studio Paris 75014,Entire place,60,1.0,Observatoire,0.942057
2531,3601285,Paris,2 pièces Paris Porte d'Orleans,Entire place,60,1.0,Observatoire,0.942057
